In [9]:
from langchain_community.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
import time
import requests
import os
from dotenv import load_dotenv

In [17]:
load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
# model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

In [11]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """
    url = f'https://v6.exchangerate-api.com/v6/{os.getenv("EXCHANGE_RATE")}/pair/{base_currency}/{target_currency}'

    response = requests.get(url)

    return response.json()['conversion_rate']

@tool
def convert(base_currency_value: int, conversion_rate: float) -> float:
    """
    given a currency conversion rate this funtion calculates the target currency value from a given base currency value
    """
    return base_currency_value*conversion_rate

In [12]:
# tool mappint
tools = {'get_conversion_factor' : get_conversion_factor, 'convert' : convert}

In [13]:
rate = get_conversion_factor.invoke({'base_currency' : 'USD', 'target_currency' : 'INR'})
print(rate)
target_value = convert.invoke({'base_currency_value': 10, 'conversion_rate' : rate})
print(target_value)

93.2612
932.6120000000001


In [19]:
llm_with_tools = model.bind_tools([get_conversion_factor, convert])

In [15]:
messages = []

messages.append(SystemMessage(content='your are a helpful chat assistant. use the provided tools to answer the user query'))
messages.append(HumanMessage(content='what is the conversion rate between USD and INR'))
messages

[SystemMessage(content='your are a helpful chat assistant. use the provided tools to answer the user query', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is the conversion rate between USD and INR', additional_kwargs={}, response_metadata={})]

In [20]:
result1 = llm_with_tools.invoke(messages)
messages.append(result1)
messages

[SystemMessage(content='your are a helpful chat assistant. use the provided tools to answer the user query', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is the conversion rate between USD and INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "INR", "base_currency": "USD"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--019d9128-f402-7a11-9457-0e3b9eabdc84-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': 'f5adb495-2ff8-4de3-b32c-79d12865a098', 'type': 'tool_call'}], usage_metadata={'input_tokens': 167, 'output_tokens': 26, 'total_tokens': 246, 'input_token_details': {'cache_read': 0}})]

In [21]:
result1.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'target_currency': 'INR', 'base_currency': 'USD'},
  'id': 'f5adb495-2ff8-4de3-b32c-79d12865a098',
  'type': 'tool_call'}]

In [22]:
for tool in result1.tool_calls:
    messages.append(tools[tool['name']].invoke(tool))
messages

[SystemMessage(content='your are a helpful chat assistant. use the provided tools to answer the user query', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is the conversion rate between USD and INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "INR", "base_currency": "USD"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--019d9128-f402-7a11-9457-0e3b9eabdc84-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': 'f5adb495-2ff8-4de3-b32c-79d12865a098', 'type': 'tool_call'}], usage_metadata={'input_tokens': 167, 'output_tokens': 26, 'total_tokens': 246, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='93.2612', name='get_conversion_factor', tool_call_id='f5adb495-2ff8

In [23]:
messages.append(HumanMessage(content='what is 50 USD in INR'))
messages

[SystemMessage(content='your are a helpful chat assistant. use the provided tools to answer the user query', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is the conversion rate between USD and INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "INR", "base_currency": "USD"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--019d9128-f402-7a11-9457-0e3b9eabdc84-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': 'f5adb495-2ff8-4de3-b32c-79d12865a098', 'type': 'tool_call'}], usage_metadata={'input_tokens': 167, 'output_tokens': 26, 'total_tokens': 246, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='93.2612', name='get_conversion_factor', tool_call_id='f5adb495-2ff8

In [24]:
result2 = llm_with_tools.invoke(messages)
messages.append(result2)
messages

[SystemMessage(content='your are a helpful chat assistant. use the provided tools to answer the user query', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is the conversion rate between USD and INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "INR", "base_currency": "USD"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--019d9128-f402-7a11-9457-0e3b9eabdc84-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': 'f5adb495-2ff8-4de3-b32c-79d12865a098', 'type': 'tool_call'}], usage_metadata={'input_tokens': 167, 'output_tokens': 26, 'total_tokens': 246, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='93.2612', name='get_conversion_factor', tool_call_id='f5adb495-2ff8

In [25]:
for tool in result2.tool_calls:
    messages.append(tools[tool['name']].invoke(tool))
messages

[SystemMessage(content='your are a helpful chat assistant. use the provided tools to answer the user query', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is the conversion rate between USD and INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "INR", "base_currency": "USD"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--019d9128-f402-7a11-9457-0e3b9eabdc84-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': 'f5adb495-2ff8-4de3-b32c-79d12865a098', 'type': 'tool_call'}], usage_metadata={'input_tokens': 167, 'output_tokens': 26, 'total_tokens': 246, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='93.2612', name='get_conversion_factor', tool_call_id='f5adb495-2ff8

In [26]:
messages.append(llm_with_tools.invoke(messages))
messages

[SystemMessage(content='your are a helpful chat assistant. use the provided tools to answer the user query', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is the conversion rate between USD and INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "INR", "base_currency": "USD"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--019d9128-f402-7a11-9457-0e3b9eabdc84-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': 'f5adb495-2ff8-4de3-b32c-79d12865a098', 'type': 'tool_call'}], usage_metadata={'input_tokens': 167, 'output_tokens': 26, 'total_tokens': 246, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='93.2612', name='get_conversion_factor', tool_call_id='f5adb495-2ff8

In [27]:
while(True):
    messages = []
    messages.append(SystemMessage(content='your are a helpful chat assistant. use the provided tools to answer the user query'))

    user_input = input('User: ')
    if user_input == 'exit':
        break
    
    print(f"User: {user_input}")
    messages.append(HumanMessage(content=user_input))
    result = llm_with_tools.invoke(messages)

    messages.append(result)

    while(result.tool_calls != []):
        for tool in result.tool_calls:
            messages.append(tools[tool['name']].invoke(tool))
        time.sleep(30)
        result = llm_with_tools.invoke(messages)

    print(f"AI: {result.content}")

User: hi, what can you
AI: Hi! I can help you with currency conversions. I can provide conversion factors between currencies and convert a base currency value to a target currency value given a conversion rate. What would you like to do? 
User: what is the conversion ration from USD to INR
AI: The conversion ratio from USD to INR is 93.2612.
User: then how much 25 USD amount in INR


KeyboardInterrupt: 